# D2 E014 - Prelink local experiment

Notebook pengembangan lokal untuk mengekstrak link biru dari screenshot resmi, menjalankan validasi target-disjoint, dan membuat CSV eksperimen. **Ini bukan notebook final E002 yang sudah diaudit.**

Perkiraan full extraction pada mesin benchmark: sekitar 90 menit. Hasil setiap shard disimpan sebagai checkpoint, jadi proses dapat dilanjutkan setelah kernel/VS Code terhenti. Semua checkpoint dibuat dari ZIP resmi oleh notebook ini sendiri.

## 0. Persiapan environment

Pilih kernel Python yang sudah memiliki dependency pada `task2/requirements-e014.txt`. Dari terminal repository, instal sekali dengan:

```powershell
python -m pip install -r task2/requirements-e014.txt
```

Bila ZIP tidak berada di folder Downloads, set environment variable `TASK2_ZIP_PATH` sebelum menjalankan notebook.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'task2').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook from inside the datathon-2026 repository')

REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / 'task2' / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from run_prelink_extract import resolve_zip_path, run as run_extraction, source_article_ids
from run_prelink_submission import build_submission, evaluate_prelink
from submission_validator import resolve_task2_data_dir

DATA_ROOT = resolve_task2_data_dir(repo_root=REPO_ROOT)
ZIP_PATH = resolve_zip_path()
WORK_DIR = REPO_ROOT / 'task2' / 'data' / 'derived' / 'd2-e014-prelink'
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
LINK_JSON = WORK_DIR / 'prelink-links.json'
SUBMISSION_PATH = REPO_ROOT / 'task2' / 'submissions' / 'sub-e014-local.csv'

WORKERS = min(4, os.cpu_count() or 1)
SHARD_PAGES = 128
LIMIT = None  # None = seluruh 4,312 current pages; isi 20 untuk smoke test saja
REUSE_COMPLETE_EXTRACTION = True

print(f'repo       : {REPO_ROOT}')
print(f'data       : {DATA_ROOT}')
print(f'official ZIP: {ZIP_PATH}')
print(f'workers    : {WORKERS}')
print(f'link JSON  : {LINK_JSON}')
print(f'submission : {SUBMISSION_PATH}')

## 1. Preflight

Pastikan `LIMIT=None` untuk full experiment. Smoke test dengan limit hanya memeriksa extractor dan **sengaja tidak dapat** diteruskan ke validasi/submission.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import PIL
import rapidocr_onnxruntime
import sklearn

required_files = [
    'states_train.csv', 'states_test.csv', 'articles.csv',
    'categories.csv', 'sample_submission.csv',
]
missing = [name for name in required_files if not (DATA_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing official files: {missing}')

article_ids = source_article_ids(DATA_ROOT)
print(f'current pages required: {len(article_ids):,}')
print(f'checkpoint shards     : {(len(article_ids) + SHARD_PAGES - 1) // SHARD_PAGES}')
print(f'numpy={np.__version__}, pandas={pd.__version__}, opencv={cv2.__version__}')
print(f'pillow={PIL.__version__}, sklearn={sklearn.__version__}')
if LIMIT is not None:
    print('WARNING: LIMIT aktif; hasil hanya smoke test dan tidak boleh dianggap submission.')

## 2. Ekstraksi link screenshot

Cell ini yang lama. Progress akan muncul sebagai `prelink pages X/Y`. Jika terputus, jalankan ulang: shard yang sudah selesai dibaca dari checkpoint. File final baru ditulis setelah seluruh shard selesai.

In [ ]:
expected_ids = source_article_ids(DATA_ROOT)
payload = None
if REUSE_COMPLETE_EXTRACTION and LINK_JSON.is_file():
    candidate = json.loads(LINK_JSON.read_text(encoding='utf-8'))
    if candidate.get('article_ids') == expected_ids and LIMIT is None:
        payload = candidate
        print('Reusing complete extraction:', LINK_JSON)

if payload is None:
    started = time.perf_counter()
    payload = run_extraction(
        data_root=DATA_ROOT,
        zip_path=ZIP_PATH,
        workers=WORKERS,
        shard_pages=SHARD_PAGES,
        limit=LIMIT,
        resume_dir=CHECKPOINT_DIR,
    )
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    LINK_JSON.write_text(
        json.dumps(payload, separators=(',', ':')) + '\n', encoding='utf-8'
    )
    print(f'Extraction cell elapsed: {time.perf_counter() - started:.1f}s')

print(json.dumps(payload['diagnostics'], indent=2))

## 3. Official local validation

Menggunakan lima fold `d2-targetgroup-v1` yang sama dengan E002. Jangan menilai E014 dari sampled diagnostic lama. Cell akan gagal tertutup bila ekstraksi belum lengkap.

In [ ]:
validation_summary = evaluate_prelink(DATA_ROOT, LINK_JSON)
fold_table = pd.DataFrame(validation_summary['folds']).rename(columns={
    'e002_accuracy': 'E002 accuracy',
    'e014_accuracy': 'E014 accuracy',
    'gain': 'E014 - E002',
})
display(fold_table)
print(f"E002 local mean : {validation_summary['e002_mean_accuracy']:.6f}")
print(f"E014 local mean : {validation_summary['e014_mean_accuracy']:.6f}")
print(f"Mean gain       : {validation_summary['mean_gain']:+.6f}")
print(f"Fold wins      : {validation_summary['fold_wins']}/5")
print(f"E014 worst fold: {validation_summary['e014_worst_fold']:.6f}")
print('Core gates     :', validation_summary['partial_gate_status'])

if validation_summary['partial_gate_status'] != 'PASS':
    print('VERDICT: REJECT pada gate inti. CSV tetap dapat dibuat untuk diagnosis, tetapi jangan submit.')
else:
    print('VERDICT: gate inti PASS; masih perlu audit leakage, subset, distribution, dan reproduksi.')

## 4. Buat dan validasi CSV lokal

CSV disimpan terpisah dari final E002 dan tidak otomatis diunggah ke Kaggle.

In [ ]:
submission_summary = build_submission(DATA_ROOT, LINK_JSON, SUBMISSION_PATH)
print(json.dumps(submission_summary, indent=2))
print('\nLocal CSV ready at:', SUBMISSION_PATH)

## Interpretasi

Notebook selesai secara teknis jika extraction lengkap, validasi tercetak, dan summary CSV berstatus `READY`. Status `READY` hanya berarti kontrak file valid; keputusan model tetap mengikuti hasil fold dan audit independen. Jangan mengganti notebook final E002 hanya karena notebook ini berhasil membuat CSV.